In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/working/checkpoints'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import zipfile

ckpt = "/kaggle/working/checkpoints/simclr-epoch=28-val_loss=0.49.ckpt"
zip_path = "/kaggle/working/simclr_best.zip"

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write(ckpt, arcname="simclr.ckpt")

print("Zipped checkpoint saved to:", zip_path)

In [ ]:
from IPython.display import FileLink
FileLink(r'simclr_best.zip')

In [ ]:
!pip install pytorch-lightning==2.3.0 lightly==1.5.0 timm==1.0.9 torchvision imagehash


In [ ]:
import json, random
from collections import defaultdict
import os
import io
import hashlib
import requests
from PIL import Image
from tqdm import tqdm
import statistics
import matplotlib.pyplot as plt
import random
import imagehash


import torch
from torch.utils.data import Dataset
import torchvision.transforms as T
import pytorch_lightning as pl
import timm
import torch.nn as nn
import torch
import lightly
from lightly.loss import NTXentLoss
import torch.nn.functional as F
from torch.utils.data import Sampler
from sklearn.neighbors import NearestNeighbors
from sklearn.manifold import TSNE


In [ ]:
tagged = json.load(open("/kaggle/input/ontariosouthbouldering/tagged_ontario_south_bouldering_and_rock_tree.json"))

# group images by route (only kept)
route2imgs = defaultdict(list)
for url, meta in tagged.items():
    if meta["tag"] == "keep":
        route2imgs[meta["route"]].append(url)

# multi-view (≥2 images)
multiview_routes = [r for r, imgs in route2imgs.items() if len(imgs) >= 2]

# hold out 8 routes → test

random.seed(42)
test_routes = random.sample(multiview_routes, 8)
train_routes = [r for r in multiview_routes if r not in test_routes]

train_multiview_imgs = sum([route2imgs[r] for r in train_routes], [])
test_multiview_imgs  = sum([route2imgs[r] for r in test_routes], [])

# All non-multiview route images
pretrain_imgs = [url for url, meta in tagged.items() if meta["tag"] == "keep" and url not in train_multiview_imgs + test_multiview_imgs]

print("Pretrain:", len(pretrain_imgs))
print("Train multiview:", len(train_multiview_imgs))
print("Test multiview:", len(test_multiview_imgs))

In [ ]:
# --------------------------------------------------
# CONFIGURE ME
# --------------------------------------------------
OUTPUT_DIR = "data"
SIMCLR_DIR = os.path.join(OUTPUT_DIR, "simclr")
MULTIVIEW_DIR = os.path.join(OUTPUT_DIR, "multiview")

os.makedirs(SIMCLR_DIR, exist_ok=True)
os.makedirs(MULTIVIEW_DIR, exist_ok=True)

# Track hashes to detect duplicates across the whole dataset
seen_hashes = {}   # hash -> {"path": path, "url": url}

HASH_THRESHOLD = 3  # max Hamming distance for near-duplicate detection

def fname_from_url(url: str) -> str:
    h = hashlib.sha1(url.encode()).hexdigest()
    return f"{h}.jpg"

def is_duplicate(img_hash):
    """Check if the hash is close to any existing image hash."""
    for h in seen_hashes.keys():
        if img_hash - h <= HASH_THRESHOLD:
            return True, seen_hashes[h]   # return original match info
    return False, None

def download_and_save(url, out_dir):
    fname = fname_from_url(url)
    path = os.path.join(out_dir, fname)

    if os.path.exists(path):
        # already downloaded -> still need hash check for duplicates
        try:
            img_hash = imagehash.phash(Image.open(path))
            dup, info = is_duplicate(img_hash)
            if not dup:
                seen_hashes[img_hash] = {"path": path, "url": url}
                return path
            else:
                print(f"⚠ Duplicate skipped: {url} matches {info['url']}")
                return None
        except:
            return None

    try:
        resp = requests.get(url, timeout=10)
        img = Image.open(io.BytesIO(resp.content)).convert("RGB")
        img_hash = imagehash.phash(img)

        dup, info = is_duplicate(img_hash)
        if dup:
            print(f"⚠ Duplicate skipped: {url} matches {info['url']}")
            return None

        img.save(path)
        seen_hashes[img_hash] = {"path": path, "url": url}
        return path

    except Exception as e:
        print(f"❌ Failed on {url}: {e}")
        return None

def download_group(urls, out_dir):
    index = []
    for url in tqdm(urls):
        path = download_and_save(url, out_dir)
        if path is not None:
            index.append({"url": url, "path": path, "route": tagged[url]["route"]})
    return index


print("\n📥 Downloading SIMCLR set...")
simclr_index = download_group(pretrain_imgs, SIMCLR_DIR)

print("\n📥 Downloading MULTIVIEW set...")
train_multiview_index = download_group(train_multiview_imgs, MULTIVIEW_DIR)
test_multiview_index = download_group(test_multiview_imgs, MULTIVIEW_DIR)
multiview_index = train_multiview_index + test_multiview_index

# Now drop any images without a partner:
# from collections import defaultdict

# count number of images per route
counts = defaultdict(int)
for sample in multiview_index:
    counts[sample["route"]] += 1

# filter out any route that has only one image
train_multiview_index = [s for s in train_multiview_index if counts[s["route"]] > 1]
test_multiview_index = [s for s in test_multiview_index if counts[s["route"]] > 1]
multiview_index = [s for s in multiview_index if counts[s["route"]] > 1]  

# --------------------------------------------------
# Save index files
# --------------------------------------------------
with open(os.path.join(OUTPUT_DIR, "index_simclr.json"), "w") as f:
    json.dump(simclr_index, f, indent=2)

with open(os.path.join(OUTPUT_DIR, "index_multiview.json"), "w") as f:
    json.dump(multiview_index, f, indent=2)

print("\n🎉 Done!")
print(f"SIMCLR saved: {len(simclr_index)} images")
print(f"MULTIVIEW saved: {len(multiview_index)} images")
print(f"Data stored in: {OUTPUT_DIR}/")

In [ ]:
# Path to your image repository
image_dir = "/kaggle/working/data/simclr"

# Lists to store widths and heights
widths = []
heights = []

# Walk through directory recursively
for root, _, files in os.walk(image_dir):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
            path = os.path.join(root, file)
            try:
                with Image.open(path) as img:
                    w, h = img.size
                    widths.append(w)
                    heights.append(h)
            except Exception as e:
                print(f"Failed to open {path}: {e}")

print(f"Found {len(widths)} images.")
print(f"Width: min={min(widths)}, max={max(widths)}, mean={sum(widths)/len(widths):.1f}, median={statistics.median(widths)}")
print(f"Height: min={min(heights)}, max={max(heights)}, mean={sum(heights)/len(heights):.1f}, median={statistics.median(heights)}")

# Optional: plot distribution
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.hist(widths, bins=20, color='skyblue')
plt.title("Width Distribution")
plt.xlabel("Width (px)")
plt.ylabel("Count")

plt.subplot(1,2,2)
plt.hist(heights, bins=20, color='salmon')
plt.title("Height Distribution")
plt.xlabel("Height (px)")
plt.ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
SIMCLR_TRANSFORMS = T.Compose([
    T.Resize(224),
    T.RandomResizedCrop(224, scale=(0.6,1.0), ratio=(0.8,1.25)),
    T.ColorJitter(0.4, 0.4, 0.4, 0.1),
    T.GaussianBlur(3, sigma=(0.1,2.0)),
    T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

FINETUNE_TRANSFORMS = T.Compose([
    T.Resize((224,224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

# ----------------------------------
# (A) SimCLR dataset: returns TWO AUGMENTED VIEWS OF SAME IMAGE
# ----------------------------------
class SimCLRDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]["path"]).convert("RGB")
        return SIMCLR_TRANSFORMS(img), SIMCLR_TRANSFORMS(img)

# ----------------------------------
# (B) Supervised contrastive dataset: label = route
# ----------------------------------
class SupConDataset(Dataset):
    def __init__(self, paths, route_to_id):
        self.paths = paths
        self.route_to_id = route_to_id
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        sample = self.paths[idx]
        img = Image.open(sample["path"]).convert("RGB")
        label = self.route_to_id[sample["route"]]
        return FINETUNE_TRANSFORMS(img), label

routes = sorted({item["route"] for item in multiview_index})
route_to_id = {route: i for i, route in enumerate(routes)}

In [ ]:
class encoder(pl.LightningModule):
    def __init__(self, lr=5e-4, temperature=0.1):
        super().__init__()
        self.encoder = timm.create_model("resnet50", pretrained=True, num_classes=0)
        self.proj = nn.Sequential(
            nn.Linear(2048, 2048),
            nn.ReLU(inplace=True),
            nn.Linear(2048, 128)
        )
        self.lr = lr
        self.temp = temperature
        self.criterion = NTXentLoss(temperature=temperature)

    def forward(self, x):
        h = self.encoder(x)
        z = self.proj(h)
        return nn.functional.normalize(z, dim=-1)

    def training_step(self, batch, batch_idx):
        x1, x2 = batch
        z1 = self(x1)
        z2 = self(x2)
        loss = self.criterion(z1, z2)
        self.log("simclr_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x1, x2 = batch
        z1, z2 = self(x1), self(x2)
        val_loss = self.criterion(z1, z2)
        self.log("val_loss", val_loss, on_step=False, on_epoch=True, prog_bar=True)
        return val_loss

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)


In [ ]:
from torch.utils.data import DataLoader
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",
    dirpath="checkpoints/",
    filename="simclr-{epoch:02d}-{val_loss:.2f}",
    save_top_k=1,
    mode="min",
    save_last=True  # very important for resuming
)

early_stop_callback = EarlyStopping(
    monitor="val_loss",     # metric to monitor
    patience=10,            # stop after 10 epochs with no improvement
    verbose=True,
    mode="min"              # lower val_loss is better
)

# --- PHASE 1: SimCLR ---
simclr_ds = SimCLRDataset(simclr_index)
simclr_train_loader = DataLoader(simclr_ds, batch_size=64, shuffle=True, num_workers=4)
multiview_ds = SimCLRDataset(multiview_index)
simclr_val_loader = DataLoader(multiview_ds, batch_size=64, shuffle=False, num_workers=4)

simclr_model = encoder()
trainer = pl.Trainer(
    max_epochs=200, 
    accelerator="gpu", 
    callbacks = [early_stop_callback, checkpoint_callback], 
    log_every_n_steps = 1
)
trainer.fit(simclr_model, simclr_train_loader, simclr_val_loader)

# Save pretrained encoder
torch.save(simclr_model.encoder.state_dict(), "simclr_encoder.pt")

In [ ]:
import shutil

best_ckpt = checkpoint_callback.best_model_path
print("Best checkpoint:", best_ckpt)

encoder = encoder.load_from_checkpoint(best_ckpt)
_ = encoder.eval()

# Save it
dst = "/kaggle/working/simclr.ckpt"

shutil.copy(best_ckpt, dst)
print("DONE: copied to /kaggle/working/simclr.ckpt")

# Pretraining Test: KNN

In [ ]:
# --- 1. Define transforms for evaluation (no random augmentation!) ---
EVAL_TRANSFORMS = T.Compose([
    T.Resize((224, 224)),      # or whatever you prefer
    T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

# --- 2. Generate embeddings without DataLoader ---
encoder.eval()
all_embeddings = []
all_paths = []
all_routes = []

with torch.no_grad():
    for sample in multiview_index:   # each sample is a dict {"path":..., "route":...}
        img = Image.open(sample["path"]).convert("RGB")
        x = EVAL_TRANSFORMS(img).unsqueeze(0)#.to(device)  # add batch dim
        z = encoder.encoder(x.cuda())
        all_embeddings.append(z.cpu())
        all_paths.append(sample["path"])
        all_routes.append(sample["route"])

embeddings = torch.cat(all_embeddings)
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

# KNN
knn = NearestNeighbors(n_neighbors=4, metric='cosine')
knn.fit(embeddings)
distances, indices = knn.kneighbors(embeddings)

# Fraction top-1 / top-3 matches
match_count = sum([all_routes[i] == all_routes[indices[i][1]] for i in range(len(all_routes))])
print(f"Top-1 match fraction: {match_count/len(all_routes)*100:.2f}%")
top3_matches = sum([all_routes[i] in [all_routes[j] for j in indices[i][1:]] for i in range(len(all_routes))])
print(f"Top-3 match fraction: {top3_matches/len(all_routes)*100:.2f}%")

# print("Nearest neighbors (excluding self):")
# for i, (dist, idxs) in enumerate(zip(distances, indices)):
#     neighbor_indices = idxs[1:]  # skip self
#     neighbor_distances = dist[1:]
#     neighbor_routes = [routes[j] for j in neighbor_indices]
#     print(f"({routes[i]}) -> ({neighbor_routes}), dist={neighbor_distances}")

In [ ]:
def plot_knn_from_index(paths, routes, indices, n_samples=10):
    import random
    sampled_idxs = random.sample(range(len(paths)), min(n_samples, len(paths)))
    
    for anchor_idx in sampled_idxs:
        anchor_img = Image.open(paths[anchor_idx]).convert("RGB")
        pred_idx = indices[anchor_idx][1]  # top-1 neighbor
        pred_img = Image.open(paths[pred_idx]).convert("RGB")
        
        # actual match that's not the anchor
        actual_matches = [i for i, r in enumerate(routes) if r == routes[anchor_idx] and i != anchor_idx]
        actual_idx = actual_matches[0] if actual_matches else None
        
        n_cols = 3 if actual_idx and actual_idx != pred_idx else 2
        fig, axs = plt.subplots(1, n_cols, figsize=(4*n_cols, 4))
        
        axs[0].imshow(anchor_img)
        axs[0].set_title(f"Anchor\nRoute {routes[anchor_idx]}")
        axs[0].axis('off')
        
        axs[1].imshow(pred_img)
        axs[1].set_title(f"Top-1 Pred\nRoute {routes[pred_idx]}")
        axs[1].axis('off')
        
        if n_cols == 3:
            actual_img = Image.open(paths[actual_idx]).convert("RGB")
            axs[2].imshow(actual_img)
            axs[2].set_title(f"Actual Match\nRoute {routes[actual_idx]}")
            axs[2].axis('off')
        
        plt.show()

plot_knn_from_index(all_paths, all_routes, indices, n_samples=30)

In [ ]:

# ----------------------------------
# 4. t-SNE visualization
# ----------------------------------
routes_int = [route_to_id[r] for r in all_routes]
# print(routes_int)
tsne = TSNE(n_components=2, perplexity=10, random_state=42)
emb_2d = tsne.fit_transform(embeddings)

plt.figure(figsize=(8,6))
scatter = plt.scatter(emb_2d[:,0], emb_2d[:,1], c=routes_int, cmap='tab20')
plt.legend(*scatter.legend_elements(), title="Route")
plt.title("t-SNE of SimCLR embeddings (pre-finetune)")
plt.show()

# Phase 2

In [ ]:
class PKSampler(Sampler):
    def __init__(self, route2indices, P, K):
        self.route2indices = list(route2indices.items())  # [(route, [idxs]), ...]
        self.P = P
        self.K = K

    def __iter__(self):
        # infinite stream of batches for an epoch
        routes = [r for r, idxs in self.route2indices]
        while True:
            chosen = random.sample(routes, self.P)
            batch = []
            for r in chosen:
                idxs = dict(self.route2indices)[r]
                if len(idxs) >= self.K:
                    batch += random.sample(idxs, self.K)
                else:
                    # upsample with replacement if too few
                    batch += [random.choice(idxs) for _ in range(self.K)]
            yield batch

    def __len__(self):
        return 1000  # not used heavily by DataLoader


In [ ]:
class SupConLossStable(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.tau = temperature

    def forward(self, features, labels):
        device = features.device
        features = F.normalize(features, dim=1)  # [B, D]
        B = features.shape[0]
    
        # cosine similarity
        logits = features @ features.T / self.tau  # [B, B]
    
        # mask for positives (exclude self)
        labels = labels.view(-1,1)
        pos_mask = torch.eq(labels, labels.T).float()
        diag_mask = torch.eye(B, device=device)
        pos_mask = pos_mask - diag_mask  # 1 for positives only
    
        # log_prob
        logits_max, _ = torch.max(logits, dim=1, keepdim=True)
        logits = logits - logits_max.detach()  # stability trick
        exp_logits = torch.exp(logits) * (1 - diag_mask)  # remove self
        log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-12)
    
        # mean log-likelihood over positives
        mean_log_prob_pos = (pos_mask * log_prob).sum(dim=1) / (pos_mask.sum(dim=1) + 1e-12)
    
        loss = -mean_log_prob_pos.mean()
        return loss


In [ ]:
class SupCon_Model(pl.LightningModule):
    def __init__(self, pretrained_encoder, lr=1e-5, temperature=0.07):
        super().__init__()
        self.encoder = pretrained_encoder
        self.proj = nn.Sequential(nn.Linear(2048, 128))
        self.loss_fn = SupConLossStable(temperature=temperature)
        self.lr = lr

    def forward(self, x):
        h = self.encoder(x)
        z = self.proj(h)
        return nn.functional.normalize(z, dim=-1)

    def training_step(self, batch, idx):
        x, labels = batch
        z = self(x)
        loss = self.loss_fn(z, labels)
        self.log("supcon_train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, idx):
        x, labels = batch
        z = self(x)
        loss = self.loss_fn(z, labels)
        self.log("supcon_val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay = 1e-4)


In [ ]:
# --- PHASE 2: Supervised contrastive fine-tuning ---
checkpoint_callback = ModelCheckpoint(
    monitor="supcon_val_loss",
    dirpath="checkpoints/",
    filename="supcon-{epoch:02d}-{val_loss:.2f}",
    save_top_k=1,
    mode="min",
    save_last=True  # very important for resuming
)

early_stop_callback = EarlyStopping(
    monitor="supcon_val_loss",     # metric to monitor
    patience=10,            # stop after 10 epochs with no improvement
    verbose=True,
    mode="min"              # lower val_loss is better
)

finetune_ds = SupConDataset(train_multiview_index, route_to_id)
test_ds     = SupConDataset(test_multiview_index, route_to_id)

train_loader = DataLoader(finetune_ds, batch_size=len(finetune_ds), shuffle=True, num_workers=4)
test_loader  = DataLoader(test_ds, batch_size=len(test_ds))

# Instantiat pretrained encoder from checkpoint:
# best_ckpt = checkpoint_callback.best_model_path
# print("Best checkpoint:", best_ckpt)
ckpt = torch.load(best_ckpt, map_location="cpu")
state = {k.replace("encoder.", ""): v for k, v in ckpt["state_dict"].items() if k.startswith("encoder.")}
encoder = timm.create_model("resnet50", pretrained=False, num_classes=0)
encoder.load_state_dict(state, strict=True)

supcon_model = SupCon_Model(pretrained_encoder=encoder)
trainer = pl.Trainer(max_epochs=100, accelerator="gpu", log_every_n_steps = 1)
trainer.fit(supcon_model, train_loader, test_loader)


In [ ]:
len(test_ds)